In [ ]:
# Problema: Antes de aplicar un modelo de priorización, el equipo debe decidir si las nuevas entradas se parecen a las conocidas durante el entrenamiento.

import json
import pickle
from pathlib import Path

import matplotlib
from IPython.display import display

matplotlib.use('Agg')

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.ensemble import IsolationForest

ACTIVITY_DIR = Path('..').resolve()
DATA_DIR = ACTIVITY_DIR / 'data'
SUBMISSION_DIR = ACTIVITY_DIR / 'submission'
MAX_ANOMALY_RATE = 0.15


In [ ]:
with (ACTIVITY_DIR / 'ESTIMATOR.pkl').open('rb') as file:
    estimator = pickle.load(file)

features = list(estimator.feature_names_in_)
training_inputs = pd.read_csv(DATA_DIR / 'training_inputs.csv').loc[:, features]
new_inputs = pd.read_csv(DATA_DIR / 'new_inputs.csv').loc[:, features]

features, training_inputs.shape, new_inputs.shape


In [ ]:
def assess_inputs(training_inputs, candidate_inputs):
    detector = IsolationForest(contamination=0.05, random_state=0)
    detector.fit(training_inputs)
    anomaly_rate = (detector.predict(candidate_inputs) == -1).mean()
    return anomaly_rate, anomaly_rate <= MAX_ANOMALY_RATE


shifted_inputs = new_inputs.assign(
    texture_mean=lambda dataframe: dataframe['texture_mean'] + 100
)

new_anomaly_rate, new_compatible = assess_inputs(training_inputs, new_inputs)
shifted_anomaly_rate, shifted_compatible = assess_inputs(
    training_inputs, shifted_inputs
)

pd.DataFrame(
    [
        {
            'dataset': 'new_inputs',
            'anomaly_rate': new_anomaly_rate,
            'compatible': new_compatible,
        },
        {
            'dataset': 'shifted_inputs',
            'anomaly_rate': shifted_anomaly_rate,
            'compatible': shifted_compatible,
        },
    ]
)


In [ ]:
fig, axis = plt.subplots(figsize=(7, 5))
axis.scatter(
    training_inputs['texture_mean'],
    training_inputs['compactness_mean'],
    alpha=0.35,
    label='X_train',
)
axis.scatter(
    new_inputs['texture_mean'],
    new_inputs['compactness_mean'],
    alpha=0.65,
    label='Entradas nuevas',
)
axis.scatter(
    shifted_inputs['texture_mean'],
    shifted_inputs['compactness_mean'],
    alpha=0.65,
    label='Entradas desplazadas',
)
axis.set_title('Entradas nuevas frente a X_train')
axis.set_xlabel('texture_mean')
axis.set_ylabel('compactness_mean')
axis.legend()
axis.grid(alpha=0.2)
display(fig)


In [ ]:
report = {
    'features': features,
    'max_anomaly_rate': MAX_ANOMALY_RATE,
    'new_inputs': {
        'rows': len(new_inputs),
        'anomaly_rate': float(new_anomaly_rate),
        'compatible': bool(new_compatible),
    },
    'shifted_inputs': {
        'rows': len(shifted_inputs),
        'anomaly_rate': float(shifted_anomaly_rate),
        'compatible': bool(shifted_compatible),
    },
}

SUBMISSION_DIR.mkdir(exist_ok=True)
report_path = SUBMISSION_DIR / 'input_distribution_report.json'
report_path.write_text(json.dumps(report, indent=2, ensure_ascii=False) + '\n', encoding='utf-8')

print(f'Reporte generado: {report_path.name}')
